# Song analysis for lyric videos (GPU)

<a href="https://colab.research.google.com/github/rohillavinaykrishan/rohillavinaykrishan/blob/main/notebooks/song_analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Upload a song (audio or video) and get:
- **Word-level lyric timing** from Whisper large-v3 (runs on the GPU)
- **Beat grid, tempo, energy curve and section boundaries** from librosa
- `lyrics.srt`, `lyrics.json` and `analysis.json`, zipped for download

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Install
!pip -q install faster-whisper librosa

In [ ]:
#@title 3. Settings
LANGUAGE = "hi"  #@param ["hi", "pa", "en", "ur", "auto"]
MODEL = "large-v3"  #@param ["large-v3", "medium", "small"]
NUM_SECTIONS = 6  #@param {type:"integer"}

# Optional: paste the correct lyrics, one line per on-screen line (e.g. in Roman script).
# Your spelling and line breaks are kept; timings come from Whisper's words.
LYRICS = """
"""

In [ ]:
#@title 4. Upload the song (audio or video)
from google.colab import files
uploaded = files.upload()
SRC = next(iter(uploaded))
!ffmpeg -v error -y -i "{SRC}" -ac 1 -ar 16000 audio.wav
print("Loaded", SRC)

In [ ]:
#@title 5. Transcribe with word timings
from faster_whisper import WhisperModel
import ctranslate2, librosa

device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
model = WhisperModel(MODEL, device=device, compute_type="float16" if device == "cuda" else "int8")
audio16k, _ = librosa.load("audio.wav", sr=16000)
segments, info = model.transcribe(
    audio16k,
    language=None if LANGUAGE == "auto" else LANGUAGE,
    word_timestamps=True,
    vad_filter=False,
)
lines = []
for s in segments:
    words = [{"word": w.word.strip(), "start": round(w.start, 3), "end": round(w.end, 3)} for w in s.words]
    lines.append({"start": round(s.start, 3), "end": round(s.end, 3), "text": s.text.strip(), "words": words})
    print(f"{s.start:6.2f} → {s.end:6.2f}  {s.text.strip()}")
print("Detected language:", info.language)

In [ ]:
#@title 6. Use your own lyric text (optional)
# Your words are laid over Whisper's word timings. Each of your lines is anchored to
# the nearest pause or segment start Whisper heard, so this works even when Whisper
# splits or merges lines differently from yours.
user_lines = [l.split() for l in LYRICS.strip().splitlines() if l.strip()]
heard, breaks = [], set()
for l in lines:
    breaks.add(len(heard))
    heard += l["words"]
for i in range(1, len(heard)):
    if heard[i]["start"] - heard[i - 1]["end"] > 0.25:
        breaks.add(i)

if user_lines and heard:
    n, m = sum(len(l) for l in user_lines), len(heard)
    starts, c = [], 0
    for words in user_lines:
        guess = round(c * m / n)
        near = [b for b in breaks if abs(b - guess) <= 3 and (not starts or b > starts[-1])]
        pick = min(near, key=lambda b: abs(b - guess)) if near else max(guess, starts[-1] + 1 if starts else 0)
        starts.append(min(pick, m - 1))
        c += len(words)
    new_lines = []
    for li, words in enumerate(user_lines):
        s0 = starts[li]
        s1 = starts[li + 1] if li + 1 < len(starts) else m
        span = max(s1 - s0, 1)
        out = []
        for i, w in enumerate(words):
            h = heard[min(s0 + i * span // len(words), m - 1)]
            start = h["start"] if not out or h["start"] > out[-1]["start"] else out[-1]["start"] + 0.05
            out.append({"word": w, "start": round(start, 3), "end": round(h["end"], 3)})
        for x, y in zip(out, out[1:]):
            x["end"] = y["start"]
        out[-1]["end"] = round(heard[min(s1, m) - 1]["end"], 3)
        new_lines.append({"start": out[0]["start"], "end": out[-1]["end"], "text": " ".join(words), "words": out})
    lines = new_lines
    for l in lines:
        print(f"{l['start']:6.2f} → {l['end']:6.2f}  {l['text']}")
    print(f"Laid {n} of your words over {m} heard words.")

In [ ]:
#@title 7. Beats, energy and sections
import librosa, numpy as np

y, sr = librosa.load("audio.wav", sr=22050)
duration = librosa.get_duration(y=y, sr=sr)
tempo, beats = librosa.beat.beat_track(y=y, sr=sr, units="time")
tempo = float(np.atleast_1d(tempo)[0])

rms = librosa.feature.rms(y=y)[0]
t = librosa.times_like(rms, sr=sr)
energy = [{"t": float(s), "rms": round(float(rms[(t >= s) & (t < s + 1)].mean()), 4)} for s in np.arange(0, duration, 1.0)]

chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
k = max(2, min(NUM_SECTIONS, chroma.shape[1] // 50))
bounds = librosa.segment.agglomerative(chroma, k)
bound_times = librosa.frames_to_time(bounds, sr=sr).tolist() + [duration]
sections = [{"start": round(a, 2), "end": round(b, 2)} for a, b in zip(bound_times[:-1], bound_times[1:])]

keys = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Duration {duration:.2f}s · tempo {tempo:.1f} BPM (half-time {tempo/2:.1f}) · strongest pitch class {keys[int(chroma.mean(1).argmax())]}")
print("Sections:", [(s["start"], s["end"]) for s in sections])

In [ ]:
#@title 8. Save and download
import json, zipfile

def ts(x):
    h, m, s = int(x // 3600), int(x % 3600 // 60), x % 60
    return f"{h:02d}:{m:02d}:{int(s):02d},{int(round((s - int(s)) * 1000)):03d}"

with open("lyrics.srt", "w", encoding="utf-8") as f:
    for i, l in enumerate(lines, 1):
        f.write(f"{i}\n{ts(l['start'])} --> {ts(l['end'])}\n{l['text']}\n\n")
json.dump({"source": SRC, "language": LANGUAGE, "lines": lines}, open("lyrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump({"duration": duration, "tempo": tempo, "beats": [round(float(b), 3) for b in beats],
           "energy": energy, "sections": sections}, open("analysis.json", "w"), indent=1)

with zipfile.ZipFile("song_analysis.zip", "w") as z:
    for name in ["lyrics.srt", "lyrics.json", "analysis.json"]:
        z.write(name)
files.download("song_analysis.zip")